# Lock-In Extraction: Phi-Coefficient Hallucination Detection

This notebook demonstrates the **Lock-In Extraction** pipeline for hallucination detection using Hadamard-multiplexed phi-coefficient computation.

**Core idea:** Build K document variants via span negation guided by a Hadamard design matrix. Extract atomic facts from each variant, then compute phi-coefficients (Pearson correlation between atom presence vectors and channel design vectors). The max-phi score serves as a hallucination signal — atoms causally grounded in the source have high phi (they disappear when their grounding span is negated), while hallucinated atoms have low phi (they persist regardless of which spans are negated).

**Five baselines** are compared:
1. Embedding cosine similarity (all-MiniLM-L6-v2)
2. LLM self-judge (yes/no grounding query)
3. Verbatim LCS ratio
4. Self-consistency (5-run re-extraction frequency)
5. LOO single-span negation

**Datasets:** MultiNLI (NLI pairs), FactScore (wiki_bio GPT-3 hallucinations), SummaC (SNLI proxy)

**Note:** This demo loads pre-computed results. The full pipeline requires an OpenRouter API key and costs ~$0.49 to run.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install)
_pip('aiohttp==3.10.5')
_pip('sentence-transformers==3.3.1')
_pip('tenacity==9.0.0')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')
    _pip('click>=8.0.0,<9.0.0')  # spaCy dep, pre-installed on Colab
    _pip('spacy==3.8.11')


In [ ]:
import asyncio
import gc
import json
import math
import os
import re
import resource
import sys
import time
from collections import defaultdict
from difflib import SequenceMatcher
from pathlib import Path
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import hadamard
from scipy.stats import pearsonr
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-9708ca-lock-in-extraction-detecting-hallucinate/main/iter_1/gen_art_experiment_1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'])} datasets")
for ds in data['datasets']:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")

## Config

Tunable parameters for this demo. Set to minimum values that still produce meaningful output. The original full-run values are shown in comments.

In [ ]:
# Hadamard variants per document (original: 8)
DEFAULT_K = 4  # original: 8

# Number of examples per dataset for demo (original: 80/20/20)
DEFAULT_N_NLIDOCS = 4    # original: 80
DEFAULT_N_FACTSCORE = 2  # original: 20 (topics)
DEFAULT_N_SUMMAC = 4     # original: 20

# Max spans to extract from a source document (original: 15/20)
MAX_SPANS = 6  # original: 15

# Soft presence threshold for embedding-based atom matching (original: 0.55)
PRESENCE_THRESHOLD = 0.55

# Bootstrap iterations for AUROC CI (original: 500)
N_BOOTSTRAP = 100  # original: 500

## Hadamard Design Matrix

The Hadamard matrix is the core of the multiplexing scheme. For K variants and N span channels, the K×N binary design matrix tells us which channels are "active" (kept as-is) vs "negated" in each variant. Column 0 of the full Hadamard matrix is all-ones (intercept) and is discarded; remaining columns form the channel assignments.

In [ ]:
def build_design_matrix(n_channels: int, k_target: int = 8) -> tuple:
    """Build K x N binary design matrix from Hadamard matrix."""
    K = 1
    while K < max(n_channels + 1, k_target):
        K *= 2
    K = min(K, 64)
    H = hadamard(K)  # K x K with +1/-1
    H_bin = (H + 1) // 2  # convert to 0/1
    # col 0 = all-ones (intercept), skip; take cols 1..N
    n_take = min(n_channels, K - 1)
    M = H_bin[:, 1:n_take + 1]  # K x n_take
    return M, K


def build_loo_matrix(n_channels: int) -> np.ndarray:
    """LOO design: N+1 rows, row 0 = original, row i+1 negates channel i."""
    M = np.ones((n_channels + 1, n_channels), dtype=int)
    for i in range(n_channels):
        M[i + 1, i] = 0
    return M


# Demonstrate: design matrix for 3 channels with K=4 variants
M_demo, K_demo = build_design_matrix(3, k_target=DEFAULT_K)
print(f"Design matrix shape: {M_demo.shape} (K={K_demo} variants x N=3 channels)")
print("Each row = one document variant; 1=keep span, 0=negate span")
print(M_demo)

## Span Segmentation and Negation

The source document is split into sentence-level spans using spaCy. Each span becomes a "channel." When a channel is "off" (design matrix = 0), the span is replaced with its negation — either rule-based (inserting "not" after aux/copula) or via LLM for longer spans.

In [ ]:
import spacy

_nlp = None

def get_nlp():
    global _nlp
    if _nlp is None:
        try:
            _nlp = spacy.load("en_core_web_sm")
        except OSError:
            import subprocess, sys
            subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'])
            _nlp = spacy.load("en_core_web_sm")
    return _nlp


def segment_spans(text: str, max_spans: int = MAX_SPANS) -> list:
    """Split text into sentence spans."""
    nlp = get_nlp()
    doc = nlp(text[:4000])
    spans = [s.text.strip() for s in doc.sents if len(s.text.strip()) > 10]
    return spans[:max_spans]


def group_spans_into_channels(spans: list) -> list:
    """Group coreferent spans. Default: each span = own channel; merge adjacent by entity overlap."""
    nlp = get_nlp()
    channels = [[i] for i in range(len(spans))]
    # Merge spans sharing >40% named entity tokens
    ents_per_span = []
    for span in spans:
        doc = nlp(span)
        ents = {e.text.lower() for e in doc.ents}
        words = {t.lemma_.lower() for t in doc if not t.is_stop and t.is_alpha and len(t.text) > 3}
        ents_per_span.append(ents | words)

    merged = list(range(len(spans)))  # channel id per span
    for i in range(len(spans)):
        for j in range(i + 1, min(i + 3, len(spans))):  # only adjacent
            if ents_per_span[i] and ents_per_span[j]:
                overlap = len(ents_per_span[i] & ents_per_span[j]) / len(ents_per_span[i] | ents_per_span[j])
                if overlap > 0.4:
                    old_chan = merged[j]
                    new_chan = merged[i]
                    for k in range(len(merged)):
                        if merged[k] == old_chan:
                            merged[k] = new_chan

    chan_to_spans = defaultdict(list)
    for span_idx, chan_id in enumerate(merged):
        chan_to_spans[chan_id].append(span_idx)
    return list(chan_to_spans.values())


def rule_based_negate(span: str) -> str:
    """Simple rule-based negation: insert 'not' after first aux/copula."""
    nlp = get_nlp()
    doc = nlp(span)
    insert_pos = -1
    for token in doc:
        if token.dep_ in ("aux", "auxpass", "cop") and token.lower_ not in ("not", "n't"):
            insert_pos = token.i
            break
    if insert_pos >= 0:
        tokens = [t.text_with_ws for t in doc]
        tokens.insert(insert_pos + 1, "not ")
        return "".join(tokens).strip()
    span_lower = span.strip()
    if span_lower and span_lower[0].isupper():
        return "It is not the case that " + span_lower[0].lower() + span_lower[1:]
    return "NOT: " + span


def build_variant(
    original_text: str,
    spans: list,
    channel_groups: list,
    design_row: np.ndarray,
    negated_spans: dict,
) -> str:
    """Build one document variant by replacing negated spans where design_row==0."""
    text = original_text
    for c_idx, group in enumerate(channel_groups):
        if c_idx >= len(design_row):
            break
        if design_row[c_idx] == 0:
            for span_idx in group:
                if span_idx < len(spans):
                    original_span = spans[span_idx]
                    negated = negated_spans.get(span_idx, rule_based_negate(original_span))
                    text = text.replace(original_span, negated, 1)
    return text


# Demonstrate negation on a sample sentence
sample = data['datasets'][0]['examples'][0]
source_text = sample['input'].split('\nSource: ')[1] if '\nSource: ' in sample['input'] else sample['input']

print("=== Span Segmentation Demo ===")
spans = segment_spans(source_text[:500], max_spans=MAX_SPANS)
for i, s in enumerate(spans[:4]):
    print(f"  Span {i}: {s[:80]}...")
    print(f"         → NEGATED: {rule_based_negate(s)[:80]}...")


## Phi-Coefficient Computation

Given the K×A atom presence matrix P and the K×N design matrix M, the phi-coefficient for atom `a` grounded in channel `n` is the Pearson correlation between column `n` of M and column `a` of P. The max |phi| across all channels gives the grounding signal: high = causally grounded (faithful), low = ungrounded (hallucinated).

In [ ]:
def build_presence_matrix(
    all_variant_atoms: list,
    canonical_atoms: list,
) -> np.ndarray:
    """Build K x A binary presence matrix."""
    K = len(all_variant_atoms)
    A = len(canonical_atoms)
    atom_to_idx = {a: i for i, a in enumerate(canonical_atoms)}
    P = np.zeros((K, A), dtype=np.float32)
    for k, atom_set in enumerate(all_variant_atoms):
        for atom in atom_set:
            if atom in atom_to_idx:
                P[k, atom_to_idx[atom]] = 1.0
    return P


def compute_phi_matrix(
    presence_matrix: np.ndarray,
    design_matrix: np.ndarray,
) -> tuple:
    """Compute phi (correlation) between design cols and atom presence.
    Returns: phi (N x A), hallucination_score (A,), provenance_span (A,)
    """
    K, A = presence_matrix.shape
    K2, N = design_matrix.shape
    assert K == K2, f"K mismatch: {K} vs {K2}"

    phi = np.zeros((N, A), dtype=np.float32)
    for n in range(N):
        col_n = design_matrix[:, n].astype(float)
        if col_n.std() < 1e-8:
            continue
        for a in range(A):
            col_a = presence_matrix[:, a]
            if col_a.std() < 1e-8:
                continue
            # Pearson correlation = phi for binary variables
            phi[n, a] = float(np.corrcoef(col_n, col_a)[0, 1])

    max_phi = np.max(np.abs(phi), axis=0)  # A-vector: max |phi| across spans
    hallucination_score = 1.0 - max_phi
    provenance = np.argmax(np.abs(phi), axis=0)  # grounding span index
    return phi, hallucination_score, provenance


# Synthetic demonstration of phi computation
print("=== Phi-Coefficient Demo (Synthetic) ===")
K, N, A = 8, 3, 4
M_syn, _ = build_design_matrix(N, k_target=K)

# Atom 0: strongly correlated with channel 0 (faithful)
# Atom 1: weakly correlated (hallucinated)
# Atom 2: correlated with channel 1
# Atom 3: no correlation (hallucinated)
rng = np.random.default_rng(42)
P_syn = np.zeros((K, A))
P_syn[:, 0] = M_syn[:, 0].astype(float)          # atom 0 tracks channel 0 perfectly
P_syn[:, 1] = rng.integers(0, 2, K).astype(float) # atom 1: random (hallucinated)
P_syn[:, 2] = M_syn[:, 1].astype(float)           # atom 2 tracks channel 1
P_syn[:, 3] = np.zeros(K)                          # atom 3: never appears

phi_syn, hall_scores, prov = compute_phi_matrix(P_syn.astype(np.float32), M_syn)

for a in range(A):
    label = "faithful" if hall_scores[a] < 0.5 else "hallucinated"
    print(f"  Atom {a}: max_phi={1-hall_scores[a]:.3f}, lockin_score={hall_scores[a]:.3f} → {label}")

## Baseline Methods

Five baselines are implemented for comparison:
1. **Embedding similarity** — max cosine between atom and source spans
2. **Verbatim LCS** — longest common subsequence ratio
3. **LLM self-judge** — yes/no grounding query (pre-computed)
4. **Self-consistency** — re-extraction frequency across K runs
5. **LOO** — leave-one-out: negate each channel and check if atom disappears

In [ ]:
from sentence_transformers import SentenceTransformer

_embedder = None

def get_embedder():
    global _embedder
    if _embedder is None:
        print("Loading sentence-transformers all-MiniLM-L6-v2...")
        _embedder = SentenceTransformer("all-MiniLM-L6-v2")
        print("Embedder loaded")
    return _embedder


def atom_match_score(atom: str, extracted_atoms: list, embedder) -> float:
    """Score how well 'atom' matches any extracted atom using embedding cosine."""
    if not extracted_atoms:
        return 0.0
    try:
        atom_emb = embedder.encode([atom], convert_to_numpy=True)
        ext_embs = embedder.encode(extracted_atoms, convert_to_numpy=True)
        cosines = (atom_emb @ ext_embs.T).flatten()
        return float(cosines.max())
    except Exception:
        ratios = [SequenceMatcher(None, atom.lower(), e.lower()).ratio() for e in extracted_atoms]
        return max(ratios) if ratios else 0.0


def build_presence_matrix_soft(
    all_variant_atoms: list,
    query_atoms: list,
    embedder,
    threshold: float = PRESENCE_THRESHOLD,
) -> np.ndarray:
    """Build K x A presence matrix using embedding match for each query atom."""
    K = len(all_variant_atoms)
    A = len(query_atoms)
    P = np.zeros((K, A), dtype=np.float32)
    for k, atom_set in enumerate(all_variant_atoms):
        extracted = list(atom_set)
        for a, query in enumerate(query_atoms):
            score = atom_match_score(query, extracted, embedder)
            P[k, a] = 1.0 if score >= threshold else 0.0
    return P


def baseline_embedding_similarity(atom: str, spans: list) -> tuple:
    """Max cosine similarity between atom and any source span."""
    if not spans:
        return 1.0, 0
    embedder = get_embedder()
    try:
        atom_emb = embedder.encode([atom], convert_to_numpy=True)
        span_embs = embedder.encode(spans, convert_to_numpy=True)
        cosines = (atom_emb @ span_embs.T).flatten()
        best_idx = int(cosines.argmax())
        return float(1.0 - cosines[best_idx]), best_idx
    except Exception:
        return 1.0, 0


def baseline_verbatim_quote(atom: str, spans: list) -> tuple:
    """LCS ratio between atom and each span."""
    if not spans:
        return 1.0, 0
    ratios = [SequenceMatcher(None, atom.lower(), s.lower()).ratio() for s in spans]
    best_idx = int(np.argmax(ratios))
    return float(1.0 - ratios[best_idx]), best_idx


# Load embedder (needed for soft presence matrix)
embedder = get_embedder()

## AUROC Evaluation

AUROC (Area Under the ROC Curve) measures how well each method separates faithful from hallucinated atoms. Score of 1.0 = perfect; 0.5 = random. We compute 95% bootstrap confidence intervals.

In [ ]:
def compute_auroc_with_ci(
    y_true: np.ndarray,
    y_score: np.ndarray,
    n_bootstrap: int = N_BOOTSTRAP,
) -> tuple:
    """Compute AUROC with 95% bootstrap CI."""
    if len(np.unique(y_true)) < 2:
        return float("nan"), float("nan"), float("nan")
    base_auroc = float(roc_auc_score(y_true, y_score))
    rng = np.random.default_rng(42)
    boots = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        boots.append(float(roc_auc_score(y_true[idx], y_score[idx])))
    if boots:
        ci_lo = float(np.percentile(boots, 2.5))
        ci_hi = float(np.percentile(boots, 97.5))
    else:
        ci_lo = ci_hi = base_auroc
    return base_auroc, ci_lo, ci_hi


def evaluate_dataset_results(records: list) -> dict:
    """Compute AUROC for all methods on a set of records."""
    if not records:
        return {}
    y_true = np.array([1 if r["output"] == "hallucinated" else 0 for r in records])
    methods = ["lockin", "embedding", "verbatim", "llm_judge", "self_consistency", "loo"]
    results = {"n": len(records)}
    for method in methods:
        key = f"predict_{method}"
        try:
            scores = np.array([float(r.get(key, 0.5)) for r in records])
            auroc, ci_lo, ci_hi = compute_auroc_with_ci(y_true, scores)
            results[method] = {"auroc": round(auroc, 4), "ci_95": [round(ci_lo, 4), round(ci_hi, 4)], "n": len(records)}
        except Exception as e:
            results[method] = {"auroc": None, "ci_95": [None, None], "n": len(records)}
    return results


# Evaluate pre-computed results from the loaded data
print("=== AUROC Evaluation on Pre-Computed Results ===")
auroc_results = {}
for ds in data['datasets']:
    name = ds['dataset']
    results = evaluate_dataset_results(ds['examples'])
    auroc_results[name] = results
    print(f"\n{name} (n={results.get('n',0)}):")
    for method in ["lockin", "embedding", "verbatim", "llm_judge", "self_consistency", "loo"]:
        r = results.get(method, {})
        auroc = r.get('auroc')
        if auroc is not None:
            print(f"  {method:<20}: AUROC={auroc:.4f}")

## Calibration

Calibration measures how well the predicted scores align with true probabilities. Expected Calibration Error (ECE) = average |predicted_probability - true_frequency| per bin. Platt scaling (logistic regression) and isotonic regression are applied as post-hoc calibrators.

In [ ]:
def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:
    """Expected Calibration Error."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for b in range(n_bins):
        mask = (probs >= bins[b]) & (probs < bins[b + 1])
        if mask.sum() > 0:
            acc = float(labels[mask].mean())
            conf = float(probs[mask].mean())
            ece += float(mask.sum()) / len(probs) * abs(acc - conf)
    return round(ece, 4)


def calibrate_scores(scores: np.ndarray, labels: np.ndarray) -> dict:
    """Platt + isotonic calibration with ECE."""
    if len(scores) < 10 or len(np.unique(labels)) < 2:
        return {"ece_before": None, "ece_after_platt": None, "ece_after_isotonic": None}
    try:
        X_tr, X_te, y_tr, y_te = train_test_split(
            scores.reshape(-1, 1), labels, test_size=0.3, random_state=42, stratify=labels
        )
        ece_before = compute_ece(X_te.ravel(), y_te)

        # Platt
        platt = LogisticRegression(max_iter=200).fit(X_tr, y_tr)
        probs_platt = platt.predict_proba(X_te)[:, 1]
        ece_platt = compute_ece(probs_platt, y_te)

        # Isotonic
        iso = IsotonicRegression(out_of_bounds="clip").fit(X_tr.ravel(), y_tr)
        probs_iso = iso.predict(X_te.ravel())
        ece_iso = compute_ece(probs_iso, y_te)

        return {
            "ece_before": ece_before,
            "ece_after_platt": ece_platt,
            "ece_after_isotonic": ece_iso,
        }
    except Exception as e:
        return {"ece_before": None, "ece_after_platt": None, "ece_after_isotonic": None}


# Print calibration results from full run (pre-computed)
print("Calibration results (from full run):")
cal = data['metadata'].get('calibration', {})
for k, v in cal.items():
    print(f"  {k}: {v}")

## Results: AUROC Comparison Across Datasets and Methods

Visualizing the full-run AUROC results from metadata (484 total atom-level labels across 3 datasets).

In [ ]:
methods = ["lockin", "embedding", "verbatim", "llm_judge", "self_consistency", "loo"]
method_labels = ["Lock-In\n(ours)", "Embedding\nSim", "Verbatim\nLCS", "LLM\nSelf-Judge", "Self-\nConsistency", "LOO"]
datasets_meta = ["multinli", "factscore", "summac"]
dataset_labels = ["MultiNLI\n(n=80)", "FactScore\n(n=344)", "SummaC\n(n=60)"]

# Extract AUROC from full-run metadata
auroc_table = {}
for ds_key, ds_label in zip(datasets_meta, dataset_labels):
    meta_key = f"auroc_{ds_key}"
    ds_data = data['metadata'].get(meta_key, {})
    auroc_table[ds_label] = {}
    for method in methods:
        m_data = ds_data.get(method, {})
        auroc_table[ds_label][method] = m_data.get('auroc')

# Print table
print(f"{'Method':<22}" + "".join(f"{d:>14}" for d in dataset_labels))
print("-" * (22 + 14 * 3))
for method, label in zip(methods, method_labels):
    label_clean = label.replace('\n', ' ')
    row = f"{label_clean:<22}"
    for ds_label in dataset_labels:
        val = auroc_table[ds_label].get(method)
        row += f"  {val:.4f}      " if val is not None else "  N/A         "
    print(row)

print(f"\nTotal atoms scored: {data['metadata']['total_atoms_scored']}")
print(f"Total API cost: ${data['metadata']['cost_total']:.4f}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for ax, ds_label in zip(axes, dataset_labels):
    vals = [auroc_table[ds_label].get(m) for m in methods]
    bar_colors = [colors[i] for i in range(len(methods))]
    bars = ax.bar(range(len(methods)), vals, color=bar_colors, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Random (0.5)')
    ax.set_title(ds_label, fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(method_labels, fontsize=8)
    ax.set_ylim(0.4, 1.0)
    ax.set_ylabel('AUROC' if ax == axes[0] else '')
    for bar, val in zip(bars, vals):
        if val is not None:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)

fig.suptitle('AUROC by Method and Dataset — Lock-In Extraction vs Baselines\n(Full run: 484 atom-level labels)', fontsize=13)
plt.tight_layout()
plt.savefig('auroc_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to auroc_comparison.png")